# apply(), map(), applymap() / Element-wise Operations

## Definition
These functions apply custom transformations to DataFrames and Series.

### When to Use Which
| Function | Input | Applies To | Use When |
|----------|-------|-----------|----------|
| `Series.map(func/dict)` | scalar | each element | Simple element-wise transforms |
| `Series.apply(func)` | scalar/array | each element | Same as map, slightly more flexible |
| `DataFrame.apply(func, axis)` | Series | each row or column | Row/column level operations |
| `DataFrame.applymap(func)` | scalar | each element (deprecated → use `map`) | Element-wise on whole DataFrame |
| `DataFrame.map(func)` | scalar | each element | pandas >= 2.1 replacement for applymap |

### Performance Note
> Avoid apply/map on large DataFrames when vectorized operations exist.
> Vectorized ops (numpy) are 10–1000x faster than apply() loops.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Carol', 'Dave'],
    'salary': [75000, 85000, 65000, 95000],
    'score':  [85.5, 72.0, 91.3, 68.8],
    'dept':   ['Eng', 'Mkt', 'Eng', 'HR']
})
print(df)

In [ ]:
# Series.map() — element-wise using dict or function
dept_code = {'Eng': 1, 'Mkt': 2, 'HR': 3}
df['dept_code'] = df['dept'].map(dept_code)
print('dept_code:', df['dept_code'].values)

# map with function
df['score_letter'] = df['score'].map(
    lambda x: 'A' if x >= 90 else ('B' if x >= 80 else ('C' if x >= 70 else 'D'))
)
print('score_letter:', df['score_letter'].values)

In [ ]:
# Series.apply() — same as map for scalar functions
df['salary_k'] = df['salary'].apply(lambda x: f'{x//1000}k')
print('salary_k:', df['salary_k'].values)

In [ ]:
# DataFrame.apply() — apply function to each row (axis=1) or column (axis=0)
# axis=0: function receives a Series for each COLUMN
col_stats = df[['salary', 'score']].apply(lambda col: col.max() - col.min(), axis=0)
print('Col-wise range:')
print(col_stats)

# axis=1: function receives a Series for each ROW
def flag_row(row):
    return 'High' if row['salary'] > 80000 and row['score'] > 80 else 'Normal'

df['flag'] = df.apply(flag_row, axis=1)
print('\nRow-wise flag:')
print(df[['name', 'salary', 'score', 'flag']])

In [ ]:
# DataFrame.apply() returning multiple values (expand=True)
def score_stats(col):
    return pd.Series({'mean': col.mean(), 'std': col.std(), 'range': col.max()-col.min()})

stats = df[['salary', 'score']].apply(score_stats, axis=0)
print('Multi-value apply result:')
print(stats)

In [ ]:
# DataFrame.map() (pandas >= 2.1, replaces applymap)
# Applies a function to every SCALAR element of the DataFrame
df_num = df[['salary', 'score']].copy()
result = df_num.map(lambda x: round(x / 100, 2))
print('Element-wise /100:')
print(result)

In [ ]:
# Performance comparison: apply vs vectorized
import time
N = 100_000
s = pd.Series(np.random.randn(N))

# apply — slow (Python loop)
t0 = time.time()
r1 = s.apply(lambda x: 1 / (1 + np.exp(-x)))
print(f'apply:      {time.time()-t0:.4f}s')

# Vectorized — fast
t0 = time.time()
r2 = 1 / (1 + np.exp(-s))
print(f'vectorized: {time.time()-t0:.4f}s')
print('Same result:', np.allclose(r1, r2))

In [ ]:
# EXCEPTION: apply result shape mismatch
df2 = df[['salary', 'score']].copy()
try:
    # Returning mismatched length inside apply
    bad = df2.apply(lambda row: pd.Series([row['salary'], row['score'], 'extra']), axis=1)
    print(bad)  # Actually works — adds column
except Exception as e:
    print(type(e).__name__, e)